# C1 CPU reward diagnostics (8 Nov 2026)

**VALIDATION ONLY**: 400 AndroidControl C1 prompts with 8 sampled S1/101
completions each; 249 tap steps carry cached UGround executability,
151 other steps have only a *text action-type proxy*.

Tests reward variants R0 historical, R0 capped-length, fixed SPICE+chrF,
R2 sparse fallback, R3 hard action gate, R4 sparse+action gate. All
proposed variants are hypotheses, not trained models. Zero-tuple fallback
uses SPICE(gold, gold) == 0, which is a proxy for parser sparsity.

Action gate *abstains* when reference text contradicts the trajectory
or the candidate action is unknown. Reports subgroup action coverage and
gold self-score; misleading comparisons are not reported as model wins.

Do not use this validation ranking alone to claim that a reward improves
downstream executability after GRPO training. Select finalists then
evaluate with controlled training seeds and a separate holdout.


In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def run(cmd):
    print("+", " ".join(map(str,cmd)), flush=True)
    subprocess.run(cmd, check=True)
run([sys.executable, "-m", "pip", "install", "-q", "pycocoevalcap", "sacrebleu"])
from pycocoevalcap.spice import spice
from pycocoevalcap.tokenizer import ptbtokenizer
libs = Path(spice.__file__).resolve().parent / "lib"
libs.mkdir(exist_ok=True)
needed = {"stanford-corenlp-3.6.0.jar": 7270741,
          "stanford-corenlp-3.6.0-models.jar": 378116526}
for name, size in needed.items():
    have = libs / name
    if not have.is_file() or have.stat().st_size != size:
        candidates = [p for p in Path("/kaggle/input").rglob(name) if p.stat().st_size == size]
        assert candidates, f"Missing {name}: attach the private/public corenlp-360-jars dataset"
        shutil.copy2(candidates[0], have)
    assert have.stat().st_size == size, f"Incorrect CoreNLP JAR {name}"
assert shutil.which("java"), "No Java: install OpenJDK before running SPICE"
print("Java", subprocess.run(["java","-version"],capture_output=True,text=True).stderr.splitlines()[:1])
print("Dependencies and CoreNLP jars OK")


In [ ]:
import sys, subprocess, shutil, json
from pathlib import Path
data = [p.parent for p in Path("/kaggle/input").rglob("c1_mau.jsonl")
        if (p.parent/"reward_cpu_8_11.py").is_file()
        and (p.parent/"score_k8_raw.jsonl").is_file()]
assert len(data)==1, f"Expected one C1 CPU dataset, got {data}"
data=data[0]
dst=Path("/kaggle/working")
shutil.copy2(data/"reward_cpu_8_11.py", dst/"reward_cpu_8_11.py")
cmd=[sys.executable, str(dst/"reward_cpu_8_11.py"), "--data", str(data),
     "--out", str(dst/"reward_cpu_8_11"), "--batch-size", "350"]
with (dst/"reward_cpu_8_11.log").open("w") as out:
    p=subprocess.run(cmd,stdout=out,stderr=subprocess.STDOUT,text=True)
log=(dst/"reward_cpu_8_11.log").read_text(errors="replace")
print(log[-8000:])
assert p.returncode == 0, f"CPU reward audit failed with code {p.returncode}"
result=json.loads((dst/"reward_cpu_8_11"/"reward_cpu_c1.json").read_text())
assert result["metadata"]["n_prompts"] == 400
assert result["rewards"]["R0_historical"]["tap"]["n"] == 249
assert result["rewards"]["R4_sparse_action"]["non_tap"]["n"] == 151
shutil.make_archive(str(dst/"reward_cpu_8_11_results"), "zip", str(dst/"reward_cpu_8_11"))
print("CPU audit DONE; output JSON and ZIP are in /kaggle/working")


In [ ]:
import json
from pathlib import Path
d=json.loads(Path("/kaggle/working/reward_cpu_8_11/reward_cpu_c1.json").read_text())
print("C1 VAL ONLY — does not show post-training improvement")
for name, m in d["rewards"].items():
    x,y=m["tap"],m["non_tap"]
    print(f"{name:25s} tap conc={x['concordance']} tie={x['tie']} std0={x['std0']} "
          f"non-tap proxy conc={y['concordance']} std0={y['std0']}")
print("Gold zero-SPICE fraction by action",d["gold_self"])
print("Gold action verifier coverage",d["trust_coverage"])
print("Stress pairs",len(d["stress"]))
